# Verify the Window Study — three architectures on one dataset

Independent re-run of the three models on a dataset and window **you** choose, to confirm the sweep numbers.

- **A — Stateful LSTM** (window as _features_): hidden/cell state carried across TBPTT chunks; ordered, no shuffle.
- **B — Memoryless** (`sever_recurrence=True`): state re-zeroed every step; the window is the only information.
- **C — Stateless windowed LSTM (Diego)**: window unrolled as _time_ (`input_size=1`), state reset between windows, **shuffled i.i.d. mini-batches**.

Edit the **Hyperparameters** cell, then _Run All_. The last cell compares your fresh numbers against `resources/window_study/window_study_results.csv`.

> A/B and C use different, paradigm-appropriate training regimes (ordered TBPTT vs shuffled i.i.d.), exactly as in the sweep. Compare each model's own trend across windows, not absolute A/B-vs-C heights.


In [ ]:
import sys, os

# Repo root on the path so `tests...` and `custom_lstm...` import from a notebook kernel.
REPO_ROOT = "C:/Users/Luis/Documents/ML-AI-Projects/custom-lstm"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

from tests.ablation_studies.data_loader import load_data
from tests.ablation_studies.config import DataMode
from tests.ablation_studies.train import seed_everything
from custom_lstm.models.lstm_vanilla_stateful import LSTMVanillaStateful
from custom_lstm.models.lstm_vanilla import LSTMVanilla
from custom_lstm.training.tbptt import TBPTTTrainerStrategy

print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())

## Hyperparameters (edit these)


In [ ]:
# --- Dataset -------------------------------------------------------------
DATASET = "aqua_alta"  # change me: aqua_alta | lbl_tcp_3 | mackey_glass | ...
DATA_DIR = os.path.join(REPO_ROOT, "data", "preprocessed")

# --- The knob under test -------------------------------------------------
WINDOW_SIZE = 16  # try 1, 2, 4, 8, 16, 32, 64, 128

# --- Shared hyperparameters (match the sweep to reproduce its numbers) ----
HIDDEN_SIZE = 64
OUTPUT_SIZE = 1
BPTT_STEPS = 50  # A/B only (TBPTT chunk length)
LR = 0.001
EPOCHS = 200
PATIENCE = 20
BATCH_SIZE_C = 512  # C only (stateless mini-batch)
SEED = 42
DEVICE = torch.device("cpu")  # CPU is faster for batch=1 recurrent stepping

# Chronological split (identical to tests/ablation_studies/data_loader.py)
TRAIN_RATIO, VAL_RATIO = 0.7, 0.15

CSV_PATH = os.path.join(DATA_DIR, f"{DATASET}.csv")
assert os.path.exists(CSV_PATH), f"not found: {CSV_PATH}"
print(f"Dataset={DATASET}  window={WINDOW_SIZE}  hidden={HIDDEN_SIZE}  epochs={EPOCHS}  patience={PATIENCE}")

## Data

**A/B** use the harness loader — window stacked as _features_, one ordered sequence `[1, N-window, window]`.
**C** builds windows as _time_ — many independent samples `[num_windows, window, 1]`.


In [ ]:
# ---- A/B data: harness windowed layout  X = [1, N-window, window] ----
splits, scaler = load_data(CSV_PATH, window_size=WINDOW_SIZE, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO)
ab_tr, ab_va = splits.train.get_by_mode(DataMode.WINDOWED), splits.val.get_by_mode(DataMode.WINDOWED)
Xtr_ab, Ytr_ab = ab_tr.X.to(DEVICE), ab_tr.Y.to(DEVICE)
Xval_ab, Yval_ab = ab_va.X.to(DEVICE), ab_va.Y.to(DEVICE)
print("A/B  X_train", tuple(Xtr_ab.shape), " X_val", tuple(Xval_ab.shape))


# ---- C data: Diego layout  X = [num_windows, window, 1] (window as time) ----
def make_windows_as_time(series, window):
    if len(series) <= window + 1:
        raise ValueError(f"window {window} too large for split of length {len(series)}")
    w = np.lib.stride_tricks.sliding_window_view(series, window)
    X = w[:-1][..., None].astype(np.float32)  # [num, window, 1]
    Y = series[window:].reshape(-1, 1).astype(np.float32)
    return torch.from_numpy(X), torch.from_numpy(Y)


def build_diego(csv_path, window):
    raw = pd.read_csv(csv_path).dropna().reset_index(drop=True).iloc[:, 0].values
    n = len(raw)
    tr_end = int(n * TRAIN_RATIO)
    val_end = tr_end + int(n * VAL_RATIO)
    sc = StandardScaler()
    tr = sc.fit_transform(raw[:tr_end].reshape(-1, 1)).flatten()
    va = sc.transform(raw[tr_end:val_end].reshape(-1, 1)).flatten()
    return make_windows_as_time(tr, window), make_windows_as_time(va, window)


(Xtr_c, Ytr_c), (Xval_c_cpu, Yval_c_cpu) = build_diego(CSV_PATH, WINDOW_SIZE)
Xval_c, Yval_c = Xval_c_cpu.to(DEVICE), Yval_c_cpu.to(DEVICE)
print("C    X_train", tuple(Xtr_c.shape), " X_val", tuple(Xval_c.shape))

## A — Stateful LSTM (window as features)

State carried across TBPTT chunks. Reports the best-epoch (early-stopped) validation MSE — the same number the sweep logs.


In [ ]:
seed_everything(SEED)
model_A = LSTMVanillaStateful(input_size=WINDOW_SIZE, hidden_size=HIDDEN_SIZE, output_size=OUTPUT_SIZE)
opt_A = torch.optim.Adam(model_A.parameters(), lr=LR)
trainer_A = TBPTTTrainerStrategy(model_A, opt_A, nn.MSELoss(), DEVICE, bptt_steps=BPTT_STEPS)
best_A = trainer_A.train(EPOCHS, Xtr_ab, Ytr_ab, Xval_ab, Yval_ab, patience=PATIENCE)
print(f"\nA stateful    best val MSE = {best_A:.6f}")

## B — Memoryless baseline (`sever_recurrence=True`)

Same setup as A, but state is re-zeroed every timestep → no recurrence. The window (as features) is the only information the model sees.


In [ ]:
seed_everything(SEED)
model_B = LSTMVanillaStateful(input_size=WINDOW_SIZE, hidden_size=HIDDEN_SIZE, output_size=OUTPUT_SIZE, sever_recurrence=True)
opt_B = torch.optim.Adam(model_B.parameters(), lr=LR)
trainer_B = TBPTTTrainerStrategy(model_B, opt_B, nn.MSELoss(), DEVICE, bptt_steps=BPTT_STEPS)
best_B = trainer_B.train(EPOCHS, Xtr_ab, Ytr_ab, Xval_ab, Yval_ab, patience=PATIENCE)
print(f"\nB memoryless  best val MSE = {best_B:.6f}")

## C — Stateless windowed LSTM (Diego)

Window unrolled as time (`input_size=1`), state reset between windows, trained with **shuffled i.i.d. mini-batches** — the way stateless models are trained (Keras `fit(shuffle=True)`).


In [ ]:
class StatelessWindowedLSTM(LSTMVanilla):
    """LSTMVanilla is abstract (no reset_state). Stateless -> reset is a no-op."""

    def reset_state(self):
        pass


seed_everything(SEED)
model_C = StatelessWindowedLSTM(input_size=1, hidden_size=HIDDEN_SIZE, output_size=OUTPUT_SIZE).to(DEVICE)
opt_C = torch.optim.Adam(model_C.parameters(), lr=LR)
crit = nn.MSELoss()
gen = torch.Generator().manual_seed(SEED)
loader = DataLoader(TensorDataset(Xtr_c, Ytr_c), batch_size=BATCH_SIZE_C, shuffle=True, generator=gen)

best_C, best_state, no_improve = float("inf"), None, 0
for epoch in range(1, EPOCHS + 1):
    model_C.train()
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        pred, _ = model_C(xb)
        loss = crit(pred, yb)
        opt_C.zero_grad()
        loss.backward()
        opt_C.step()
    model_C.eval()
    with torch.no_grad():
        vpred, _ = model_C(Xval_c)
        vloss = crit(vpred, Yval_c).item()
    if vloss < best_C:
        best_C, best_state, no_improve = vloss, copy.deepcopy(model_C.state_dict()), 0
    else:
        no_improve += 1
    if epoch == 1 or epoch % 20 == 0:
        print(f"  epoch {epoch:>3}/{EPOCHS}  val MSE {vloss:.6f}")
    if no_improve >= PATIENCE:
        print(f"  early stop @ epoch {epoch}")
        break
print(f"\nC stateless   best val MSE = {best_C:.6f}")

## Compare with the sweep

Your fresh numbers vs the logged `window_study_results.csv` for this dataset & window.
Small last-digit differences are normal across processes; magnitudes and trends should match.


In [ ]:
fresh = {"lstm_vanilla_windowed": best_A, "lstm_vanilla_windowed_no_recurrence": best_B, "lstm_vanilla_stateless_windowed": best_C}
labels = {
    "lstm_vanilla_windowed": "A stateful",
    "lstm_vanilla_windowed_no_recurrence": "B memoryless",
    "lstm_vanilla_stateless_windowed": "C stateless (Diego)",
}

csv_path = os.path.join(REPO_ROOT, "resources", "window_study", "window_study_results.csv")
ref = pd.read_csv(csv_path) if os.path.exists(csv_path) else None

rows = []
for arch, val in fresh.items():
    row = {"model": labels[arch], "this_notebook": round(val, 6), "sweep_csv": None, "abs_diff": None}
    if ref is not None:
        m = ref[(ref.dataset == DATASET) & (ref.window_size == WINDOW_SIZE) & (ref.architecture == arch)]
        if len(m):
            sv = float(m.best_val_loss.iloc[0])
            row["sweep_csv"] = round(sv, 6)
            row["abs_diff"] = round(abs(sv - val), 6)
    rows.append(row)

print(f"Dataset = {DATASET}   window = {WINDOW_SIZE}\n")
print(pd.DataFrame(rows).to_string(index=False))